<a href="https://colab.research.google.com/github/pancakexia/machinelearning/blob/pancakexia_project/huggingface_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipdb
import ipdb

In [ ]:
!python -m ipdb your_script.py --your-arg value

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install torch


In [ ]:
!pip install -U transformers datasets evaluate accelerate timm

In [ ]:
!pip install transformers datasets torch torchvision


In [ ]:
!apt install git-lfs

In [ ]:
from datasets import load_dataset

# 加载 coco2017 数据集
dataset = load_dataset("mlx-vision/imagenet-1k", split="train")

# 打印前几条数据
print(dataset[0])


In [ ]:
# 图像预处理
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整图像大小
    transforms.ToTensor(),          # 转换为 Tensor 格式
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 用ImageNet的均值和标准差进行归一化
])

# 选择一张图像进行测试
image_path = dataset[0]['coco_url']
image = Image.open(image_path)

# 应用预处理
input_image = transform(image).unsqueeze(0)  # 增加批次维度

In [ ]:
!mkdir -p /content/coco2017
!wget -q http://images.cocodataset.org/zips/train2017.zip -O /content/coco2017/train2017.zip
!unzip -q /content/coco2017/train2017.zip -d /content/coco2017

In [ ]:
import os
from PIL import Image

coco_root = "/content/coco2017"
file_name = dataset[0]["file_name"]              # e.g. "train2017/000000391895.jpg"
image_path = os.path.join(coco_root, file_name)  # "/content/coco2017/train2017/000000391895.jpg"
image = Image.open(image_path).convert("RGB")


In [ ]:
import os
from PIL import Image

# 1) 拿到相对路径
file_name = dataset[0]['file_name']
# → 'train2017/000000391895.jpg'

# 2) 本地根目录
coco_root = "/content/000000391895.jpg"

# 3) 拼出绝对路径
image_path = os.path.join(coco_root, file_name)
assert os.path.exists(image_path), f"{image_path} not found"

# 4) 打开并预处理
image = Image.open(image_path).convert("RGB")
input_image = transform(image).unsqueeze(0)

In [ ]:
/mnt/data/coco2017/train2017/000000391895.jpg


In [ ]:
from transformers import ViTForImageClassification, ViTFeatureExtractor

# 加载预训练的 Vision Transformer 模型和特征提取器
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224-in21k")
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")

# 通过特征提取器将图像处理为模型输入
inputs = feature_extractor(images=image, return_tensors="pt")


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-5)

# 假设你有标签数据
labels = torch.tensor([1])  # 示例标签，通常需要通过数据集加载实际标签

# 向前传播并计算损失
outputs = model(**inputs)
logits = outputs.logits  # 模型输出
loss = criterion(logits, labels)  # 计算损失

# 反向传播并更新模型参数
loss.backward()
optimizer.step()

print("Loss:", loss.item())



In [ ]:
from transformers import Trainer, TrainingArguments

# 设置训练参数
training_args = TrainingArguments(
    output_dir='./results',          # 输出目录
    num_train_epochs=3,              # 训练轮数
    per_device_train_batch_size=8,   # 每个设备的训练批次大小
    per_device_eval_batch_size=16,   # 每个设备的评估批次大小
    evaluation_strategy="epoch",     # 每个周期评估一次
    logging_dir='./logs',            # 日志目录
)

# 创建 Trainer
trainer = Trainer(
    model=model,                         # 预训练模型
    args=training_args,                  # 训练参数
    train_dataset=dataset,               # 训练数据集
    eval_dataset=dataset,                # 验证数据集
    compute_metrics=None,                # 计算评价指标（如果需要）
)

# 开始训练
trainer.train()


In [ ]:
results = trainer.evaluate()

print("Evaluation results:", results)


In [ ]:
# 加载测试图像
test_image = Image.open("test_image.jpg")
test_inputs = feature_extractor(test_image, return_tensors="pt")

# 使用训练后的模型进行预测
with torch.no_grad():
    outputs = model(**test_inputs)

# 获取预测结果
logits = outputs.logits
predicted_class = logits.argmax(-1).item()

print(f"Predicted class: {predicted_class}")
